# NBA Win/Loss Prediction: Sample Analysis

**Project goal:** build a machine learning model that predicts whether the home team wins an NBA game using recent team performance and matchup-differential features.

This notebook combines the main parts of the project into one polished analysis:

1. Data cleaning
2. Exploratory data analysis
3. Feature engineering
4. Model training and evaluation
5. Model comparison
6. Final interpretation and future work

The analysis focuses on modern NBA seasons from **2021–2026**. Seasons 2021–2024 are used for training, 2025 is used for testing, and 2026 is kept for future/live prediction.

## 1. Loading Libraries and Data

The first step is to load the required Python libraries and the raw NBA games dataset. The `low_memory=False` argument helps pandas inspect the dataset more carefully before assigning column data types.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve
)

# Optional XGBoost import
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

# Flexible file path: works if Games.csv is in the current folder or data/raw/
possible_paths = [
    Path("Games.csv"),
    Path("data/raw/Games.csv"),
    Path("../data/raw/Games.csv")
]

data_path = next((p for p in possible_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError("Could not find Games.csv. Put it in the notebook folder or data/raw/.")

raw_games = pd.read_csv(data_path, low_memory=False)
print("Raw data shape:", raw_games.shape)
raw_games.head()

## 2. Data Cleaning

The cleaning process prepares the dataset for time-aware sports prediction. The most important decisions are:

- Convert dates into datetime format.
- Filter to modern NBA seasons, 2021–2026.
- Remove duplicate or incomplete games.
- Create the target variable, `home_win`.
- Create readable team-name columns.
- Keep final scores in the cleaned dataset for feature engineering, but avoid using final scores as direct model predictors.

In [ ]:
games = raw_games.copy()

# Convert date columns
games["gameDateTimeEst"] = pd.to_datetime(games["gameDateTimeEst"], errors="coerce")
games["gameDate"] = pd.to_datetime(games["gameDate"], errors="coerce")

# Create season year
games["season_year"] = games["gameDateTimeEst"].dt.year

# Filter to modern seasons
games = games[(games["season_year"] >= 2021) & (games["season_year"] <= 2026)].copy()

# Remove duplicates
games = games.drop_duplicates()
games = games.drop_duplicates(subset=["gameId"])

# Remove incomplete or invalid score rows
games = games.dropna(subset=["homeScore", "awayScore"])
games = games[(games["homeScore"] > 0) & (games["awayScore"] > 0)].copy()

# Create target variable: 1 if home team won, 0 otherwise
games["home_win"] = (games["homeScore"] > games["awayScore"]).astype(int)

# Create readable team names
games["home_team"] = (
    games["hometeamCity"].astype(str) + " " + games["hometeamName"].astype(str)
).str.replace(r"\s+", " ", regex=True).str.strip()

games["away_team"] = (
    games["awayteamCity"].astype(str) + " " + games["awayteamName"].astype(str)
).str.replace(r"\s+", " ", regex=True).str.strip()

# Create contextual indicators
games["is_playoff"] = (games["gameType"] != "Regular Season").astype(int)
games["covid_era"] = (games["season_year"] <= 2021).astype(int)

# Remove rows missing team names
games = games.dropna(subset=["home_team", "away_team"])

# Sort chronologically
games = games.sort_values("gameDateTimeEst").reset_index(drop=True)

print("Cleaned data shape:", games.shape)
games[["gameId", "gameDateTimeEst", "season_year", "home_team", "away_team", "homeScore", "awayScore", "home_win", "is_playoff"]].head()

### Cleaning Check

Before modeling, I checked whether the target distribution and season counts were reasonable. This is important because unrealistic values here would signal a problem in the cleaning process.

In [ ]:
print("Home win distribution:")
print(games["home_win"].value_counts(normalize=True))

print("
Games by season:")
print(games["season_year"].value_counts().sort_index())

print("
Remaining missing values in key columns:")
print(games[["gameId", "gameDateTimeEst", "season_year", "home_team", "away_team", "home_win"]].isna().sum())

## 3. Exploratory Data Analysis

The purpose of the EDA is not only to make plots, but to verify that the dataset behaves realistically. For example, NBA home teams should win more often than away teams, and 2026 may be incomplete because it represents the current/future prediction season.

In [ ]:
# Games per season
season_counts = games["season_year"].value_counts().sort_index()
season_counts.plot(kind="bar")
plt.title("Number of NBA Games by Season")
plt.xlabel("Season Year")
plt.ylabel("Number of Games")
plt.tight_layout()
plt.show()

In [ ]:
# Home win distribution
home_win_counts = games["home_win"].value_counts(normalize=True).sort_index()
home_win_counts.index = ["Away Team Wins", "Home Team Wins"]
home_win_counts.plot(kind="bar")
plt.title("Home Win vs Away Win Distribution")
plt.xlabel("Outcome")
plt.ylabel("Proportion")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Home win rate by season
home_win_by_season = games.groupby("season_year")["home_win"].mean()
home_win_by_season.plot(kind="line", marker="o")
plt.title("Home Win Rate by Season")
plt.xlabel("Season Year")
plt.ylabel("Home Win Rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 4. Feature Engineering

This is the most important stage of the project. The raw dataset is game-centered, but rolling sports features require team-centered data. Therefore, each NBA game is converted into two rows: one from the home team's perspective and one from the away team's perspective.

This allows the model to learn each team's recent history before a matchup.

In [ ]:
# Home-team perspective
home = games[[
    "gameId", "gameDateTimeEst", "season_year",
    "home_team", "away_team", "homeScore", "awayScore",
    "home_win", "is_playoff"
]].copy()

home = home.rename(columns={
    "home_team": "team",
    "away_team": "opponent",
    "homeScore": "points_for",
    "awayScore": "points_against"
})
home["is_home"] = 1
home["win"] = home["home_win"]

# Away-team perspective
away = games[[
    "gameId", "gameDateTimeEst", "season_year",
    "home_team", "away_team", "homeScore", "awayScore",
    "home_win", "is_playoff"
]].copy()

away = away.rename(columns={
    "away_team": "team",
    "home_team": "opponent",
    "awayScore": "points_for",
    "homeScore": "points_against"
})
away["is_home"] = 0
away["win"] = 1 - away["home_win"]

team_games = pd.concat([home, away], ignore_index=True)
team_games = team_games.sort_values(["team", "gameDateTimeEst"]).reset_index(drop=True)

team_games["point_diff"] = team_games["points_for"] - team_games["points_against"]

team_games.head()

### Rolling Features

Rolling features summarize how a team performed before the current game. The key anti-leakage step is `shift(1)`, which prevents the current game's result from being included in its own predictor values.

In [ ]:
# Rolling last-5 features using only previous games
team_games["last_5_win_pct"] = (
    team_games.groupby("team")["win"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

team_games["offensive_average"] = (
    team_games.groupby("team")["points_for"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

team_games["defensive_average"] = (
    team_games.groupby("team")["points_against"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

team_games["last_5_point_diff"] = (
    team_games.groupby("team")["point_diff"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

team_games[[
    "team", "opponent", "gameDateTimeEst", "win", "point_diff",
    "last_5_win_pct", "offensive_average", "defensive_average", "last_5_point_diff"
]].head(12)

## 5. Matchup-Differential Features

After creating team-level rolling features, the data is converted back into one row per game. The model predicts whether the **home team** wins, so the final features compare the home team's recent performance against the away team's recent performance.

In [ ]:
home_feature_cols = [
    "gameId", "season_year", "home_win", "team",
    "last_5_win_pct", "offensive_average", "defensive_average", "last_5_point_diff"
]

away_feature_cols = [
    "gameId", "team",
    "last_5_win_pct", "offensive_average", "defensive_average", "last_5_point_diff"
]

home_features = team_games[team_games["is_home"] == 1][home_feature_cols].copy()
away_features = team_games[team_games["is_home"] == 0][away_feature_cols].copy()

home_features = home_features.add_prefix("home_")
away_features = away_features.add_prefix("away_")

home_features = home_features.rename(columns={
    "home_gameId": "gameId",
    "home_season_year": "season_year",
    "home_home_win": "home_win"
})

away_features = away_features.rename(columns={
    "away_gameId": "gameId"
})

matchup_data = home_features.merge(away_features, on="gameId", how="inner")

# Matchup-differential features
matchup_data["win_pct_diff"] = matchup_data["home_last_5_win_pct"] - matchup_data["away_last_5_win_pct"]
matchup_data["offense_diff"] = matchup_data["home_offensive_average"] - matchup_data["away_offensive_average"]
matchup_data["defense_diff"] = matchup_data["home_defensive_average"] - matchup_data["away_defensive_average"]
matchup_data["point_diff_diff"] = matchup_data["home_last_5_point_diff"] - matchup_data["away_last_5_point_diff"]

matchup_data[[
    "gameId", "season_year", "home_team", "away_team", "home_win",
    "win_pct_diff", "offense_diff", "defense_diff", "point_diff_diff"
]].head()

The final model features are:

- `win_pct_diff`: home team's recent win percentage minus away team's recent win percentage
- `offense_diff`: home team's recent scoring average minus away team's recent scoring average
- `defense_diff`: home team's recent points allowed minus away team's recent points allowed
- `point_diff_diff`: home team's recent point differential minus away team's recent point differential

These features summarize the relative strength of the home team compared with the away team.

## 6. Modeling Setup

Because NBA games occur over time, I used a chronological split rather than a random split. Random splitting could leak future information into the training process. The modeling structure is:

- Train: 2021–2024
- Test: 2025
- Future prediction: 2026

In [ ]:
feature_cols = [
    "win_pct_diff",
    "offense_diff",
    "defense_diff",
    "point_diff_diff"
]

model_df = matchup_data[feature_cols + ["home_win", "season_year"]].dropna().copy()

train_df = model_df[model_df["season_year"] <= 2024].copy()
test_df = model_df[model_df["season_year"] == 2025].copy()
future_2026 = model_df[model_df["season_year"] == 2026].copy()

X_train = train_df[feature_cols]
y_train = train_df["home_win"]

X_test = test_df[feature_cols]
y_test = test_df["home_win"]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("2026 future prediction shape:", future_2026.shape)

## 7. Baseline Model: Logistic Regression

Logistic regression is used as the baseline because it is interpretable and produces probability estimates. This is important because sports prediction is not only about selecting winners, but also about estimating confidence.

In [ ]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

log_preds = log_model.predict(X_test)
log_probs = log_model.predict_proba(X_test)[:, 1]

log_accuracy = accuracy_score(y_test, log_preds)
log_auc = roc_auc_score(y_test, log_probs)

print("Logistic Regression Accuracy:", log_accuracy)
print("Logistic Regression ROC-AUC:", log_auc)
print("
Classification Report:")
print(classification_report(y_test, log_preds))

In [ ]:
cm = confusion_matrix(y_test, log_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Logistic Regression Confusion Matrix")
plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, log_probs)

plt.plot(fpr, tpr, label=f"AUC = {log_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
log_coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": log_model.coef_[0]
}).sort_values("Coefficient", ascending=False)

log_coef_df.plot(x="Feature", y="Coefficient", kind="bar")
plt.title("Logistic Regression Coefficients")
plt.ylabel("Coefficient")
plt.tight_layout()
plt.show()

log_coef_df

## 8. Additional Models

To test whether more complex models improve performance, I compared logistic regression with regularized models and tree-based models:

- Ridge logistic regression
- Lasso logistic regression
- Random Forest
- XGBoost, if installed
- Tuned logistic regression

In [ ]:
# Ridge logistic regression: L2 penalty
ridge_model = LogisticRegression(penalty="l2", C=1.0, max_iter=1000)
ridge_model.fit(X_train, y_train)
ridge_preds = ridge_model.predict(X_test)
ridge_probs = ridge_model.predict_proba(X_test)[:, 1]
ridge_accuracy = accuracy_score(y_test, ridge_preds)
ridge_auc = roc_auc_score(y_test, ridge_probs)

# Lasso logistic regression: L1 penalty
lasso_model = LogisticRegression(penalty="l1", solver="liblinear", C=1.0, max_iter=1000)
lasso_model.fit(X_train, y_train)
lasso_preds = lasso_model.predict(X_test)
lasso_probs = lasso_model.predict_proba(X_test)[:, 1]
lasso_accuracy = accuracy_score(y_test, lasso_preds)
lasso_auc = roc_auc_score(y_test, lasso_probs)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=4025
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_accuracy = accuracy_score(y_test, rf_preds)
rf_auc = roc_auc_score(y_test, rf_probs)

print("Ridge Accuracy:", ridge_accuracy)
print("Ridge ROC-AUC:", ridge_auc)
print("Lasso Accuracy:", lasso_accuracy)
print("Lasso ROC-AUC:", lasso_auc)
print("Random Forest Accuracy:", rf_accuracy)
print("Random Forest ROC-AUC:", rf_auc)

In [ ]:
rf_importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

rf_importance_df.plot(x="Feature", y="Importance", kind="bar")
plt.title("Random Forest Feature Importance")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

rf_importance_df

In [ ]:
if XGBOOST_AVAILABLE:
    xgb_model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=4025,
        eval_metric="logloss",
        n_jobs=1
    )
    xgb_model.fit(X_train, y_train)
    xgb_preds = xgb_model.predict(X_test)
    xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
    xgb_accuracy = accuracy_score(y_test, xgb_preds)
    xgb_auc = roc_auc_score(y_test, xgb_probs)
    
    print("XGBoost Accuracy:", xgb_accuracy)
    print("XGBoost ROC-AUC:", xgb_auc)
    
    xgb_importance_df = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": xgb_model.feature_importances_
    }).sort_values("Importance", ascending=False)
    
    xgb_importance_df.plot(x="Feature", y="Importance", kind="bar")
    plt.title("XGBoost Feature Importance")
    plt.ylabel("Importance")
    plt.tight_layout()
    plt.show()
    
    display(xgb_importance_df)
else:
    xgb_accuracy = np.nan
    xgb_auc = np.nan
    print("XGBoost is not installed in this environment.")

## 9. Hyperparameter Tuning

Since logistic regression performed strongly as a baseline, I tuned the regularization strength using cross-validation. The main parameter is `C`, where smaller values imply stronger regularization.

In [ ]:
log_param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100]
}

log_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000),
    param_grid=log_param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

log_search.fit(X_train, y_train)

print("Best Parameters:")
print(log_search.best_params_)

print("
Best CV ROC-AUC:")
print(log_search.best_score_)

best_log = log_search.best_estimator_
best_log_preds = best_log.predict(X_test)
best_log_probs = best_log.predict_proba(X_test)[:, 1]

best_log_accuracy = accuracy_score(y_test, best_log_preds)
best_log_auc = roc_auc_score(y_test, best_log_probs)

print("
Tuned Logistic Accuracy:", best_log_accuracy)
print("Tuned Logistic ROC-AUC:", best_log_auc)

## 10. Model Comparison

The final comparison uses accuracy and ROC-AUC. Accuracy measures the share of correct classifications, while ROC-AUC measures how well the model ranks likely home wins above likely home losses. For this project, ROC-AUC is especially important because the long-term goal is probability-based sports prediction.

In [ ]:
comparison_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Ridge",
        "Lasso",
        "Random Forest",
        "XGBoost",
        "Tuned Logistic"
    ],
    "Accuracy": [
        log_accuracy,
        ridge_accuracy,
        lasso_accuracy,
        rf_accuracy,
        xgb_accuracy,
        best_log_accuracy
    ],
    "ROC_AUC": [
        log_auc,
        ridge_auc,
        lasso_auc,
        rf_auc,
        xgb_auc,
        best_log_auc
    ]
})

comparison_df

In [ ]:
comparison_df.plot(x="Model", y="Accuracy", kind="bar")
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

In [ ]:
comparison_df.plot(x="Model", y="ROC_AUC", kind="bar")
plt.title("Model ROC-AUC Comparison")
plt.ylabel("ROC-AUC")
plt.ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

## 11. Final Model for 2026 Prediction

After evaluating models on the 2025 test season, the best-performing model can be retrained on all known seasons from 2021–2025. This gives the final model the largest available historical training set before making 2026 predictions.

In [ ]:
final_train_df = model_df[model_df["season_year"] <= 2025].copy()
final_predict_df = model_df[model_df["season_year"] == 2026].copy()

X_final = final_train_df[feature_cols]
y_final = final_train_df["home_win"]

final_model = LogisticRegression(
    C=log_search.best_params_["C"],
    max_iter=1000
)

final_model.fit(X_final, y_final)

X_2026 = final_predict_df[feature_cols]
final_predict_df["predicted_home_win_prob"] = final_model.predict_proba(X_2026)[:, 1]
final_predict_df["predicted_home_win"] = final_model.predict(X_2026)

final_predict_df[["season_year", "home_win", "predicted_home_win_prob", "predicted_home_win"]].head()

## 12. Interpretation

The main result of this analysis is that carefully engineered matchup-differential features were more important than model complexity. The tuned logistic regression model performed competitively with or better than more complex models, suggesting that recent win percentage, scoring margin, offensive production, and defensive performance contain relatively stable predictive signal.

The Random Forest and XGBoost feature-importance results also showed that `point_diff_diff` was one of the most important predictors. This makes basketball sense because recent point differential captures both offensive and defensive strength in one number.

## 13. Limitations and Future Work

This analysis is a strong first version, but it has important limitations:

- It does not include player injuries.
- It does not include trades or roster changes directly.
- It does not include rest days, travel distance, or back-to-back games yet.
- It does not include sportsbook odds or implied probabilities yet.
- It uses only last-5 rolling windows; future versions could compare last-10 and last-20 windows.
- The 2026 season may be incomplete, so predictions from that season should be interpreted carefully.

Future work could improve the model by adding player-level data, injury reports, betting odds, Elo ratings, and calibration analysis. These additions would make the model more realistic for playoff and Finals prediction.

## Conclusion

This project built an end-to-end NBA win/loss prediction pipeline using modern NBA data from 2021–2026. The analysis cleaned the raw game data, engineered rolling team-strength features, created matchup-differential predictors, compared multiple machine learning models, and selected a final model for 2026 prediction.

The most important conclusion is that feature engineering drove much of the predictive performance. Simpler interpretable models performed very strongly because the engineered features already summarized meaningful basketball information. This supports the broader machine learning lesson that strong domain-specific features can be more valuable than simply using more complex algorithms.

::: callout-note
Some code debugging and writing support were completed with the help of AI. All modeling decisions, outputs, and interpretations were reviewed and understood before inclusion in the final analysis.
:::